# H2 — Gold Validation & Preparation
Notebook ini hanya sebagai antarmuka inspeksi. Logic utama berada di `src/simplexid/gold.py`.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

cwd = Path.cwd()
candidates = [cwd, *cwd.parents]
ROOT = next(
    p for p in candidates
    if (p / "src" / "simplexid").exists()
    and (p / "data").exists()
)

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("ROOT:", ROOT)
print("SRC :", SRC)


In [ ]:
from simplexid.gold import (
    validate_review_log,
    build_gold_pairs,
    validate_gold_pairs,
    save_split_files,
    build_polarity_template,
    build_implicit_template,
    h2_status,
)

print("Import H2 berhasil.")


In [ ]:
terma = pd.read_csv(
    ROOT / "data" / "processed" / "terma_processed.csv"
)

review = pd.read_csv(
    ROOT / "data" / "gold" / "gold_review_log.csv"
)

print("TermA :", terma.shape)
print("Review:", review.shape)
print(review["annotation_status"].value_counts(dropna=False))


In [ ]:
review_summary = validate_review_log(
    review,
    terma,
    require_reviewed=True,
)

review_summary


In [ ]:
gold_pairs = build_gold_pairs(review, terma)

pair_summary = validate_gold_pairs(
    gold_pairs,
    terma,
    review,
)

print(pair_summary)
gold_pairs.head(20)


In [ ]:
gold_pairs_path = (
    ROOT / "data" / "gold" / "gold_pairs.csv"
)

gold_pairs.to_csv(
    gold_pairs_path,
    index=False,
    encoding="utf-8-sig",
)

print("Saved:", gold_pairs_path)


In [ ]:
split_meta = save_split_files(
    terma,
    ROOT / "data" / "splits",
)

split_meta


In [ ]:
# Buat template polarity hanya jika file belum ada.
polarity_path = ROOT / "data" / "gold" / "gold_polarity.csv"

if not polarity_path.exists():
    polarity = build_polarity_template(
        gold_pairs,
        final_sample_size=600,
        dev_sample_size=400,
        seed=42,
    )
    polarity.to_csv(
        polarity_path,
        index=False,
        encoding="utf-8-sig",
    )
    print("Created:", polarity_path, len(polarity))
else:
    print("Existing file retained:", polarity_path)


In [ ]:
# Buat template implicit hanya jika file belum ada.
implicit_path = ROOT / "data" / "gold" / "gold_implicit.csv"

if not implicit_path.exists():
    implicit = build_implicit_template(
        terma,
        dev_sample_size=250,
        seed=42,
    )
    implicit.to_csv(
        implicit_path,
        index=False,
        encoding="utf-8-sig",
    )
    print("Created:", implicit_path, len(implicit))
else:
    print("Existing file retained:", implicit_path)


In [ ]:
h2_status(ROOT)
